# LAB 3 — NHẬN DẠNG TIẾNG NÓI NÂNG CAO VỚI HMM

## 0. Cấu hình và kiểm tra train/test

In [ ]:
from pathlib import Path
from collections import Counter, defaultdict
import os, time, json, pickle, math, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import librosa, soundfile as sf
from scipy.special import logsumexp
from scipy.spatial.distance import cdist
from hmmlearn.hmm import GaussianHMM
from sklearn.metrics import accuracy_score, confusion_matrix, ConfusionMatrixDisplay
from IPython.display import display

ROOT = Path.cwd()
DATA = ROOT / 'dataset'
FIG = ROOT / 'figures'; RESULTS = ROOT / 'results'; MODELS = ROOT / 'models'
for folder in (FIG, RESULTS, MODELS): folder.mkdir(exist_ok=True)
LABELS = ['mo','dong','len','xuong','dung']
LABEL_VI = dict(zip(LABELS, ['mở','đóng','lên','xuống','dừng']))
FS = 16000; HOP = 160; WIN = 400; N_FFT = 512
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)
print('Project:', ROOT)
print('hmmlearn:', __import__('hmmlearn').__version__)

def find_wavs(part):
    return {lab: sorted((DATA/part/lab).glob('*.wav')) for lab in LABELS}
train_files = find_wavs('train'); test_files = find_wavs('test')
counts = pd.DataFrame({'label': LABELS, 'train': [len(train_files[x]) for x in LABELS],
                       'test': [len(test_files[x]) for x in LABELS]})
display(counts)
missing = counts[(counts['train'] < 6) | (counts['test'] < 2)]
if not missing.empty:
    raise FileNotFoundError('Chưa đủ WAV! Thêm ít nhất 6 train + 2 test cho mỗi lớp vào dataset/train/<label> và dataset/test/<label>. Xem README.md.')
# Hash file để bắt trường hợp một file được copy sang cả train/test
import hashlib
def sha256_file(path):
    h = hashlib.sha256()
    with open(path, 'rb') as fh:
        for chunk in iter(lambda: fh.read(1024*1024), b''): h.update(chunk)
    return h.hexdigest()
tr_hash = {sha256_file(p) for arr in train_files.values() for p in arr}
te_hash = {sha256_file(p) for arr in test_files.values() for p in arr}
assert not (tr_hash & te_hash), 'Data leakage: cùng nội dung WAV xuất hiện ở train và test!'
print('OK: đủ file và không có WAV trùng bytes.')
manifest_path = DATA / 'audio_manifest.csv'
if manifest_path.exists():
    manifest = pd.read_csv(manifest_path)
    print('Manifest của bộ dữ liệu TTS mẫu (nếu thay WAV thì cần cập nhật manifest):')
    display(pd.crosstab(manifest['label'], manifest['split']))
    print('Tổng WAV ghi trong manifest:', len(manifest))
manifest_path = DATA / 'audio_manifest.csv'
if manifest_path.exists():
    manifest = pd.read_csv(manifest_path)
    print('Manifest của bộ dữ liệu TTS mẫu (nếu thay WAV thì cần cập nhật manifest):')
    display(pd.crosstab(manifest['label'], manifest['split']))
    print('Tổng WAV ghi trong manifest:', len(manifest))

## A. Audio preprocessing và MFCC 39 chiều
Tiền nhấn 0.97, Hamming 25 ms/10 ms, 13 MFCC + 13 Δ + 13 Δ². CMVN *riêng từng utterance* giống case study mục 5.2, thống nhất train/test. Việc chuẩn hóa từng utterance không dùng thống kê của tập test để tính chuẩn hóa trên tập train.

In [ ]:
def load_audio(path):
    y, _ = librosa.load(str(path), sr=FS, mono=True)
    if len(y) < WIN:
        raise ValueError(f'Âm thanh quá ngắn: {path}')
    return y.astype(np.float32)

def preprocess_audio(y, trim=True, top_db=30):
    if trim:
        yt, _ = librosa.effects.trim(y, top_db=top_db, frame_length=WIN, hop_length=HOP)
        if len(yt) >= WIN: y = yt
    return librosa.effects.preemphasis(y, coef=0.97)

def extract_feature(path_or_audio, dynamic=True, trim=True, return_parts=False):
    y = load_audio(path_or_audio) if isinstance(path_or_audio, (str, Path)) else np.asarray(path_or_audio)
    y = preprocess_audio(y, trim=trim)
    mfcc = librosa.feature.mfcc(y=y, sr=FS, n_mfcc=13, n_fft=N_FFT,
        win_length=WIN, hop_length=HOP, window='hamming', n_mels=26,
        fmin=20, fmax=FS/2, center=False)
    if mfcc.shape[1] < 9: raise ValueError('Quá ít frame cho delta: cần ít nhất 9 frame')
    d1 = librosa.feature.delta(mfcc, order=1)
    d2 = librosa.feature.delta(mfcc, order=2)
    blocks = [mfcc, d1, d2] if dynamic else [mfcc]
    X = np.vstack(blocks).T
    mu = X.mean(axis=0, keepdims=True)
    sigma = X.std(axis=0, keepdims=True)
    X = (X-mu)/np.maximum(sigma, 1e-5)
    return (X, (mfcc,d1,d2)) if return_parts else X

for lab in LABELS[:3]:
    p = train_files[lab][0]
    print(p.relative_to(ROOT), '=>', extract_feature(p).shape)

for lab in LABELS[:2]:
    X, parts = extract_feature(train_files[lab][0], return_parts=True)
    fig, axs = plt.subplots(3, 1, figsize=(11, 7), constrained_layout=True)
    for ax, M, name in zip(axs, parts, ['MFCC','Delta MFCC','Delta2 MFCC']):
        im = ax.imshow(M, origin='lower', aspect='auto', extent=(0,M.shape[1]*HOP/FS,0,13))
        ax.set(title=f'{LABEL_VI[lab]} — {name}', xlabel='Thời gian (s)', ylabel='Hệ số')
        fig.colorbar(im, ax=ax)
    fig.savefig(FIG/f'features_{lab}.png', dpi=150)
    plt.show()

## B. Gaussian HMM trái sang phải
`π=[1,0,...]`, chỉ có self-loop và sang trạng thái kế. Để giữ topology **bất biến**, đặt `params='mc'`: Baum–Welch vẫn cập nhật Gaussian phát xạ nhưng **không cập nhật A**; như vậy đây là biến thể constrained EM (không phải tối ưu chuyển trạng thái A). Có thể thử bật `params='tmc'` nhưng cần kiểm tra mọi transition bị cấm sau fit.

In [ ]:
def make_lr_hmm(n_states=5, n_iter=80, covariance_type='diag'):
    model = GaussianHMM(n_components=n_states, covariance_type=covariance_type,
        n_iter=n_iter, tol=1e-3, min_covar=1e-3,
        init_params='mc', params='mc', random_state=RANDOM_STATE,
        implementation='log')
    start = np.zeros(n_states); start[0] = 1.0
    A = np.zeros((n_states,n_states))
    for i in range(n_states-1): A[i,i], A[i,i+1] = 0.7, 0.3
    A[-1,-1] = 1.0
    model.startprob_ = start
    model.transmat_ = A
    return model

def validate_topology(model):
    A = model.transmat_; n=len(A)
    allowed = np.eye(n, dtype=bool) | np.eye(n, k=1, dtype=bool)
    assert np.all(A[~allowed] < 1e-12), 'Có transition nhảy lùi hoặc skip'
    assert np.allclose(A.sum(axis=1), 1), 'Tổng xác suất hàng không bằng 1'
    assert np.allclose(model.startprob_, np.r_[1., np.zeros(n-1)])
    return True

example = make_lr_hmm()
print('Initial pi:', example.startprob_)
print('Initial A:\n', example.transmat_)
assert validate_topology(example)

## C. Huấn luyện mỗi từ một HMM
Ghép feature theo cùng lớp thành `X_concat` và truyền `lengths`; theo dõi convergence và lưu `.pkl`. Không đưa test vào huấn luyện.

In [ ]:
feature_cache = {}
def feat(path, dynamic=True):
    k = (str(path), dynamic)
    if k not in feature_cache: feature_cache[k] = extract_feature(path, dynamic=dynamic)
    return feature_cache[k]

def train_hmms(train_map, n_states=5, dynamic=True, covariance_type='diag', save=False):
    models = {}; training_rows = []
    start_time = time.perf_counter()
    for lab in LABELS:
        feats = [feat(p, dynamic) for p in train_map[lab]]
        X = np.vstack(feats); lengths = [len(a) for a in feats]
        m = make_lr_hmm(n_states=n_states, covariance_type=covariance_type)
        with warnings.catch_warnings(record=True) as caught:
            warnings.simplefilter('always')
            m.fit(X,lengths)
        validate_topology(m)
        training_rows.append({'label':lab,'n_states':n_states,'iterations':m.monitor_.iter,
            'converged':m.monitor_.converged,'train_loglik':m.score(X,lengths),
            'warnings':'; '.join(str(w.message) for w in caught)})
        models[lab] = m
        if save:
            with open(MODELS/f'hmm_{lab}.pkl','wb') as f:pickle.dump(m,f)
    return models, pd.DataFrame(training_rows), time.perf_counter()-start_time

models, training_log, hmm_train_sec = train_hmms(train_files, n_states=5, save=True)
display(training_log)
training_log.to_csv(RESULTS/'training_log.csv', index=False)
for lab, m in models.items():
    print(f'{lab} pi=', m.startprob_, '\nA=\n', np.round(m.transmat_,3))

## D. Acoustic likelihood và Viterbi state alignment
`model.score(X)` là Forward log-likelihood; `decode(...,algorithm='viterbi')` tìm một path trạng thái tốt nhất. Hai khái niệm khác nhau. Xuất bảng score tất cả model và 2 hình Viterbi.

In [ ]:
def recognize_hmm(X, fitted):
    scores = {lab: float(m.score(X)) for lab,m in fitted.items()}
    pred=max(scores,key=scores.get)
    return pred, scores

sample_wav = test_files[LABELS[0]][0]
Xsample = feat(sample_wav)
pred, scores = recognize_hmm(Xsample,models)
display(pd.DataFrame([{'model':k,'log_likelihood':v,'log_likelihood_per_frame':v/len(Xsample)} for k,v in scores.items()]).sort_values('log_likelihood',ascending=False))
print('Prediction:',LABEL_VI[pred])
for lab in LABELS[:2]:
    wav=test_files[lab][0]; X=feat(wav)
    score, state_path=models[lab].decode(X,algorithm='viterbi')
    duration=Counter(state_path)
    print(wav.name, 'Viterbi:',round(score,2), 'frames/state:',dict(sorted(duration.items())))
    plt.figure(figsize=(10,3))
    plt.step(np.arange(len(state_path))*HOP/FS, state_path+1, where='post')
    plt.yticks(range(1,models[lab].n_components+1))
    plt.xlabel('Thời gian (s)');plt.ylabel('State');plt.title(f'Viterbi alignment: {wav.name}')
    plt.tight_layout();plt.savefig(FIG/f'viterbi_{lab}.png',dpi=150);plt.show()

## H. Đánh giá trên tập test sạch
Confusion matrix; top-1/top-2 acoustic scores; phân phối correct-vs-incorrect log-likelihood/frame để gợi ý reject threshold **không hiệu chỉnh bằng test**.

In [ ]:
def evaluate_hmm(fitted, file_map, dynamic=True, prefix='hmm', plot=True):
    out=[]; t0=time.perf_counter()
    for true_lab in LABELS:
        for wav in file_map[true_lab]:
            X=feat(wav, dynamic); pred,scores=recognize_hmm(X,fitted)
            ranked=sorted(scores.items(),key=lambda x:x[1],reverse=True)
            out.append({'file':str(wav.relative_to(ROOT)),'true':true_lab,'pred':pred,'correct':int(pred==true_lab),
                'frames':len(X),'top1_score':ranked[0][1], 'top2_score':ranked[1][1],
                'top1_per_frame':ranked[0][1]/len(X),
                'true_score_per_frame':scores[true_lab]/len(X),
                'best_wrong_per_frame':max(s/len(X) for l,s in scores.items() if l!=true_lab),
                'margin_per_frame':(ranked[0][1]-ranked[1][1])/len(X)})
    df=pd.DataFrame(out); accuracy=accuracy_score(df.true,df.pred)
    df.to_csv(RESULTS/f'{prefix}_predictions.csv',index=False)
    if plot:
        cm=confusion_matrix(df.true,df.pred,labels=LABELS)
        disp=ConfusionMatrixDisplay(cm,display_labels=[LABEL_VI[l] for l in LABELS])
        disp.plot(cmap='Blues',values_format='d')
        plt.title(f'{prefix}: accuracy={accuracy:.1%}');plt.tight_layout()
        plt.savefig(FIG/f'{prefix}_confusion.png',dpi=150);plt.show()
    return df, accuracy, (time.perf_counter()-t0)/len(df)

hmm_df, hmm_acc, hmm_test_sec=evaluate_hmm(models,test_files)
print(f'HMM accuracy={hmm_acc:.2%}; test time/trial={hmm_test_sec:.4f}s')
display(hmm_df)
plt.figure(figsize=(8,4))
plt.hist(hmm_df.true_score_per_frame,bins=12,alpha=.6,label='Đúng lớp')
plt.hist(hmm_df.best_wrong_per_frame,bins=12,alpha=.6,label='Sai lớp tốt nhất')
plt.xlabel('Log-likelihood/frame');plt.ylabel('Số lượng');plt.legend();plt.tight_layout()
plt.savefig(FIG/'score_distributions.png',dpi=150);plt.show()
print('Reject threshold phải chọn trên validation, KHÔNG dựa vào test scores. Có thể dùng max score/frame và margin/frame.')

## E. Baseline DTW (Lab 2) — cùng train/test split

In [ ]:
def dtw_norm(A,B):
    C=cdist(A,B,'euclidean'); n,m=C.shape
    D=np.full((n+1,m+1),np.inf);L=np.zeros((n+1,m+1),dtype=np.int32)
    D[0,0]=0
    for i in range(1,n+1):
        for j in range(1,m+1):
            options=(D[i-1,j],D[i,j-1],D[i-1,j-1])
            k=int(np.argmin(options)); pi,pj=((i-1,j),(i,j-1),(i-1,j-1))[k]
            D[i,j]=C[i-1,j-1]+options[k]
            L[i,j]=L[pi,pj]+1
    return D[n,m]/max(L[n,m],1)

def recognize_dtw(X,template_map):
    scores={lab:min(dtw_norm(X,ref) for ref in refs) for lab,refs in template_map.items()}
    return min(scores,key=scores.get),scores

t0=time.perf_counter()
templates={lab:[feat(p,dynamic=False) for p in train_files[lab]] for lab in LABELS}
dtw_train_sec=time.perf_counter()-t0
rows=[];t0=time.perf_counter()
for lab in LABELS:
    for wav in test_files[lab]:
        pred,scores=recognize_dtw(feat(wav,dynamic=False),templates)
        ranked=sorted(scores.items(),key=lambda x:x[1])
        rows.append({'file':str(wav.relative_to(ROOT)),'true':lab,'pred':pred,'top1_distance':ranked[0][1],'top2_distance':ranked[1][1]})
dtw_df=pd.DataFrame(rows);dtw_test_sec=(time.perf_counter()-t0)/len(dtw_df)
dtw_acc=accuracy_score(dtw_df.true,dtw_df.pred)
dtw_df.to_csv(RESULTS/'dtw_predictions.csv',index=False)
disp=ConfusionMatrixDisplay(confusion_matrix(dtw_df.true,dtw_df.pred,labels=LABELS),display_labels=[LABEL_VI[l] for l in LABELS])
disp.plot(cmap='Blues');plt.title(f'DTW: {dtw_acc:.1%}');plt.tight_layout();plt.savefig(FIG/'dtw_confusion.png',dpi=150);plt.show()
comparison=pd.DataFrame([
 {'model':'DTW (13 MFCC)','accuracy':dtw_acc,'train_sec':dtw_train_sec,'test_sec_per_trial':dtw_test_sec},
 {'model':'HMM (39 MFCC)','accuracy':hmm_acc,'train_sec':hmm_train_sec,'test_sec_per_trial':hmm_test_sec}])
display(comparison);comparison.to_csv(RESULTS/'dtw_vs_hmm.csv',index=False)
paired=hmm_df[['file','correct','pred']].merge(dtw_df[['file','pred']],on='file',suffixes=('_hmm','_dtw'))
paired['dtw_correct']=(paired['pred_dtw']==paired['file'].map(dict(zip(dtw_df.file,dtw_df.true))))
display(paired[(paired.correct.astype(bool)!=paired.dtw_correct)])

## F. Mini lexicon
Trong whole-word HMM **không cần** từ điển phát âm. Tệp `resources/lexicon.txt` là **template trống có chú thích**, không tự bịa phoneme tiếng Việt. Khi được cung cấp bộ phone inventory, điền `WORD PHONE_1 PHONE_2...` rồi parser chạy được.

In [ ]:
def parse_lexicon(path):
    out={}
    for line_no,line in enumerate(Path(path).read_text(encoding='utf-8').splitlines(),1):
        line=line.split('#',1)[0].strip()
        if not line:continue
        parts=line.split()
        if len(parts)<2:raise ValueError(f'Lexicon line {line_no}: thiếu phones')
        if parts[0] in out:raise ValueError(f'Trùng WORD: {parts[0]}')
        out[parts[0]]=parts[1:]
    return out
lexicon=parse_lexicon(ROOT/'resources'/'lexicon.txt')
print('Lexicon entries:',lexicon if lexicon else '(chưa có inventory; cần tự nhập)')

## G. Bigram + MAP re-ranking
Corpus trong `resources/corpus.txt` là **corpus tạo thủ công để minh họa thuật toán**, không phản ánh phân bố ngôn ngữ thực tế. Bạn có thể thay bằng corpus do giảng viên cung cấp. Nếu bỏ trống, phần LM sẽ thông báo bỏ qua. Add-k smoothing dùng counts đúng theo lịch sử `prev`, thử λ=0, 0.1, 0.3, 1.0. Acoustic score chia cho số frame để thuận tiện chọn λ.

In [ ]:
def load_corpus(path):
    sentences=[]
    for line in Path(path).read_text(encoding='utf-8').splitlines():
        line=line.split('#',1)[0].strip().lower()
        if not line:continue
        toks=line.split()
        unknown=set(toks)-set(LABELS)
        if unknown:raise ValueError(f'Corpus có nhãn không thuộc vocabulary: {unknown}')
        sentences.append(toks)
    return sentences

def build_bigram(sentences):
    c1=Counter();c2=Counter()
    for sentence in sentences:
        seq=['<s>']+sentence
        for prev,cur in zip(seq[:-1],seq[1:]):c1[prev]+=1;c2[(prev,cur)]+=1
    return c1,c2

def bigram_prob(prev,cur,c1,c2,V,k=0.5):
    return (c2[(prev,cur)]+k)/(c1[prev]+k*V)

def rerank(scores,frames,prev,c1,c2,lm_weight):
    total={lab:s/frames+lm_weight*math.log(bigram_prob(prev,lab,c1,c2,len(LABELS)))
           for lab,s in scores.items()}
    return max(total,key=total.get), total

sentences=load_corpus(ROOT/'resources'/'corpus.txt')
if not sentences:
    print('Chưa có corpus thực tế -> chưa thể làm thí nghiệm LM; bổ sung resources/corpus.txt rồi Run All.')
else:
    c1,c2=build_bigram(sentences)
    candidates=[]
    for lab in LABELS:
        for wav in test_files[lab]:
            X=feat(wav);pred,scores=recognize_hmm(X,models)
            ranked=sorted(scores.values(),reverse=True)
            candidates.append((abs((ranked[0]-ranked[1])/len(X)),wav,X,scores))
    _,selected,X,scores=min(candidates,key=lambda row:row[0])
    # Previous word được chỉ định cho mô phỏng, không giả định đó là ngữ cảnh có thật của test WAV.
    prev='<s>'; rows=[]
    for w in [0,0.1,0.3,1.0]:
        prediction,total=rerank(scores,len(X),prev,c1,c2,w)
        rows.append({'file':selected.name,'prev_word_assumed':prev,'lambda_lm':w,'pred':prediction,
                     'best_combined_score':total[prediction]})
    lm_df=pd.DataFrame(rows);display(lm_df);lm_df.to_csv(RESULTS/'lm_rerank.csv',index=False)

## E1–E3. Thí nghiệm mô hình có kiểm soát
E1: 3/5/7 states, cùng feature 39-D. E2: 13-D và 39-D, cùng 5 states. E3: diag so với full (chỉ chạy nếu người dùng bật, vì nguy cơ suy biến ở dữ liệu ít). Không tự động chọn mô hình tốt nhất dựa trên test.

In [ ]:
experiment_rows=[]
for states in (3,5,7):
    fitted, log, sec=train_hmms(train_files,n_states=states,dynamic=True)
    _,acc,avg=evaluate_hmm(fitted,test_files,True,prefix=f'e1_states_{states}',plot=False)
    experiment_rows.append({'experiment':'E1_states','setting':str(states),'accuracy':acc,
                            'train_sec':sec,'test_sec_per_trial':avg,'converged_all':bool(log.converged.all())})
for dynamic in (False,True):
    fitted,log,sec=train_hmms(train_files,n_states=5,dynamic=dynamic)
    _,acc,avg=evaluate_hmm(fitted,test_files,dynamic,prefix=f'e2_dynamic_{dynamic}',plot=False)
    experiment_rows.append({'experiment':'E2_features','setting':'39D' if dynamic else '13D',
                            'accuracy':acc,'train_sec':sec,'test_sec_per_trial':avg,'converged_all':bool(log.converged.all())})
RUN_E3_FULL=False  # Chỉ bật khi có đủ dữ liệu cho covariance đầy đủ 39x39/state
if RUN_E3_FULL:
    fitted,log,sec=train_hmms(train_files,5,True,'full')
    _,acc,avg=evaluate_hmm(fitted,test_files,True,prefix='e3_full',plot=False)
    experiment_rows.append({'experiment':'E3_covariance','setting':'full','accuracy':acc,
                            'train_sec':sec,'test_sec_per_trial':avg,'converged_all':bool(log.converged.all())})
else:
    print('E3 full covariance đã tắt mặc định do tập nhỏ; diag có trong baseline.')
exps=pd.DataFrame(experiment_rows);display(exps);exps.to_csv(RESULTS/'experiments_E1_E3.csv',index=False)

## E4–E6 và H — robustness/mismatch
E4 là bảng `dtw_vs_hmm.csv` ở phần E. E5: dữ liệu TTS đã có sẵn **nhiễu/vang mô phỏng** trong `dataset/robustness/<label>/*.wav`. Đây là controlled synthetic mismatch, không phải ghi âm môi trường thực và không thuộc tập huấn luyện. E6 là λLM trong phần G (chỉ chạy khi có corpus).

In [ ]:
robust_files=find_wavs('robustness')
if any(robust_files.values()):
    robust_df,robust_acc,_=evaluate_hmm(models,robust_files,True,prefix='e5_robustness',plot=True)
    print(f'E5: clean {hmm_acc:.2%} -> robustness {robust_acc:.2%}; drop={hmm_acc-robust_acc:+.2%}')
else:
    print('E5 chưa chạy: hãy bổ sung WAV bị mismatch vào dataset/robustness/<label>/')
print('E4 kết quả trong results/dtw_vs_hmm.csv; E6 kết quả trong results/lm_rerank.csv nếu đã thêm corpus.')

## Dự đoán file WAV mới (demo nhận dạng)
Nhập đường dẫn file WAV bất kỳ vào `UNKNOWN_WAV`. Nếu trống, cell chỉ hướng dẫn mà không đưa ra dự đoán giả.

In [ ]:
UNKNOWN_WAV = ''  # Ví dụ: 'my_new_recording.wav'
if UNKNOWN_WAV and Path(UNKNOWN_WAV).is_file():
    X=extract_feature(UNKNOWN_WAV)
    pred,scores=recognize_hmm(X,models)
    display(pd.DataFrame(sorted(scores.items(),key=lambda a:a[1],reverse=True),columns=['label','log_likelihood']))
    print('Kết quả:',LABEL_VI[pred])
else:
    print('Điền UNKNOWN_WAV bằng đường dẫn file âm thanh WAV để thử nhận dạng.')